# 두 메소드 비교 평가 (언어 compliance 필터링 적용)

이 노트북은 지정한 두 메소드(Method A와 Method B)에 대해 **질문 언어와 두 메소드의 응답 언어가 모두 일치하는 데이터**만 선별하여 수학 평가 정확도(Accuracy)를 비교 평가합니다.

**지원 벤치마크**: PolyMath, MATH-500, MGSM

In [1]:
MODEL_PATH = "Meta-Llama-3.1-8B-Instruct"
# MODEL_PATH = "Meta-Llama-3.1-70B-Instruct"
# MODEL_PATH = "Qwen2.5-7B-Instruct"
# MODEL_PATH = "Qwen2.5-14B-Instruct"
# MODEL_PATH = "Qwen2.5-72B-Instruct"

# 결과가 저장된 경로 설정
PRJ_PATH = "/home/work/mlp/hslim/LASEF2/data/results/llama"

# FastText 언어 감지 모델 경로
FASTTEXT_MODEL_PATH = "/home/work/mlp/hslim/LASEF2/lid.176.bin"

def make_path(prj_path, benchmark, model_path, suffix):
    return f"{prj_path}/{benchmark}/{model_path}{suffix}.jsonl"
    
# 비교할 두 메소드의 suffix
SUFFIX_A = "-cot"
SUFFIX_B = "-skeleton_multiturn"
# SUFFIX_A = "-cot-Google-transQ"
# SUFFIX_B = "-skeleton_multiturn-Google-transQ"

In [2]:
import json
import os
import fasttext
import traceback
from tqdm import tqdm
from multiprocessing import Pool, cpu_count
from collections import defaultdict
from math_verify import parse, verify

# FastText 모델 로드
fasttext.FastText.eprint = lambda x: None  # 경고 출력 비활성화
lang_model = fasttext.load_model(FASTTEXT_MODEL_PATH)
print("FastText model loaded successfully.")

def detect_language(text):
    """FastText를 사용하여 언어를 감지합니다."""
    if text is None or not isinstance(text, str):
        return "unk"
    text = text.replace("\n", " ").strip()
    if len(text) < 2:
        return "unk"
    try:
        labels, probs = lang_model.predict(text, k=1)
        if not probs or len(probs) == 0:
            return "unk"
        return labels[0].replace("__label__", "")
    except:
        return "unk"

def load_jsonl(filepath):
    """JSONL 파일을 읽어 데이터 리스트로 반환합니다."""
    data = []
    if not os.path.exists(filepath):
        print(f"⚠️ File not found: {filepath}")
        return None
    with open(filepath, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"JSON error on line {i}: {e}")
    return data

def get_response(example):
    """responses[0] 또는 response 키에서 모델 출력 텍스트를 가져옵니다."""
    if 'responses' in example and isinstance(example['responses'], list) and len(example['responses']) > 0:
        return example['responses'][0]
    elif 'response' in example and isinstance(example['response'], str):
        return example['response']
    return ""

def evaluate_example(args):
    """한 샘플의 정답 여부(1 또는 0)를 판별합니다."""
    resp, answer = args
    try:
        gold = parse(str(answer))
        pred = parse(str(resp))
        is_correct = verify(gold, pred)
        return int(is_correct), 0
    except Exception:
        return 0, 1

def evaluate_and_compare(benchmark_name, prj_path, model_path, suffix_a, suffix_b):
    """
    지정된 두 메소드의 결과를 로드하고, 질문 언어와 두 응답 언어가 모두 일치하는 경우만 채점합니다.
    """
    path_a = make_path(prj_path, benchmark_name, model_path, suffix_a)
    path_b = make_path(prj_path, benchmark_name, model_path, suffix_b)
    
    print(f"\n============================================================")
    print(f"📊 Benchmark: {benchmark_name}")
    print(f"============================================================")
    print(f"   Method A: {os.path.basename(path_a)}")
    print(f"   Method B: {os.path.basename(path_b)}")
    
    data_a = load_jsonl(path_a)
    data_b = load_jsonl(path_b)
    
    if data_a is None or data_b is None:
        print("❌ Skip: 데이터 로드 실패")
        return
    
    print(f"   Loaded samples: A={len(data_a)}, B={len(data_b)}")
    
    # 샘플 개수 통일
    min_len = min(len(data_a), len(data_b))
    data_a = data_a[:min_len]
    data_b = data_b[:min_len]
    
    lang_groups = defaultdict(list)
    mismatch_a_count = 0
    mismatch_b_count = 0
    both_match_count = 0
    
    print("🔍 Filtering language compliance pairs...")
    for item_a, item_b in tqdm(zip(data_a, data_b), total=min_len, desc="Filtering"):
        q_lang = item_a.get('question_language', 'unknown')
        if isinstance(q_lang, (list, tuple)):
            q_lang = q_lang[0] if len(q_lang) > 0 else 'unknown'
            
        resp_a = get_response(item_a)
        resp_b = get_response(item_b)
        
        lang_a = detect_language(resp_a)
        lang_b = detect_language(resp_b)
        
        if lang_a != q_lang:
            mismatch_a_count += 1
        if lang_b != q_lang:
            mismatch_b_count += 1
            
        if lang_a == q_lang and lang_b == q_lang:
            both_match_count += 1
            lang_groups[q_lang].append((resp_a, resp_b, item_a.get('answer')))
            
    print(f"   Filtering Results:")
    print(f"     - Total pairs: {min_len}")
    print(f"     - Method A mismatch: {mismatch_a_count}")
    print(f"     - Method B mismatch: {mismatch_b_count}")
    print(f"     - Both match (Evaluated): {both_match_count} ({both_match_count/min_len*100:.1f}%)")
    
    if both_match_count == 0:
        print("⚠️ 평가할 수 있는 매칭 샘플이 없습니다.")
        return
        
    # 병렬 평가용 인수 설정
    eval_args = []
    for lang, pairs in lang_groups.items():
        for resp_a, resp_b, ans in pairs:
            eval_args.append((resp_a, ans))
            eval_args.append((resp_b, ans))
            
    print(f"   Evaluating accuracies using {cpu_count()} workers...")
    with Pool(cpu_count()) as pool:
        raw_results = pool.map(evaluate_example, eval_args)
        
    # 평가 점수 매핑 및 집계
    idx = 0
    lang_scores = defaultdict(lambda: {'correct_a': 0, 'correct_b': 0, 'total': 0, 'errors_a': 0, 'errors_b': 0})
    overall_correct_a = 0
    overall_correct_b = 0
    overall_total = 0
    
    for lang, pairs in lang_groups.items():
        for resp_a, resp_b, ans in pairs:
            res_a, err_a = raw_results[idx]
            res_b, err_b = raw_results[idx+1]
            idx += 2
            
            lang_scores[lang]['correct_a'] += res_a
            lang_scores[lang]['correct_b'] += res_b
            lang_scores[lang]['errors_a'] += err_a
            lang_scores[lang]['errors_b'] += err_b
            lang_scores[lang]['total'] += 1
            
            overall_correct_a += res_a
            overall_correct_b += res_b
            overall_total += 1
            
    print("\n📈 =================== COMPARISON SUMMARY ===================")
    for lang, scores in sorted(lang_scores.items()):
        tot = scores['total']
        cor_a = scores['correct_a']
        cor_b = scores['correct_b']
        acc_a = cor_a / tot * 100
        acc_b = cor_b / tot * 100
        delta = acc_b - acc_a
        print(f"   [{lang.upper()}] Acc A={acc_a:.2f}% ({cor_a}/{tot}) | Acc B={acc_b:.2f}% ({cor_b}/{tot}) | Delta={delta:+.2f}%")
        
    print("   ----------------------------------------------------------")
    ovr_acc_a = overall_correct_a / overall_total * 100
    ovr_acc_b = overall_correct_b / overall_total * 100
    ovr_delta = ovr_acc_b - ovr_acc_a
    print(f"   [OVERALL] Acc A={ovr_acc_a:.2f}% ({overall_correct_a}/{overall_total}) | Acc B={ovr_acc_b:.2f}% ({overall_correct_b}/{overall_total}) | Delta={ovr_delta:+.2f}%")
    print("============================================================\n")

FastText model loaded successfully.


## 1. PolyMath Comparison

In [3]:
evaluate_and_compare(
    benchmark_name="PolyMath-translated",
    prj_path=PRJ_PATH,
    model_path=MODEL_PATH,
    suffix_a=SUFFIX_A,
    suffix_b=SUFFIX_B
)


📊 Benchmark: PolyMath-translated
   Method A: Meta-Llama-3.1-8B-Instruct-cot.jsonl
   Method B: Meta-Llama-3.1-8B-Instruct-skeleton_multiturn.jsonl
   Loaded samples: A=3500, B=3500
🔍 Filtering language compliance pairs...


Filtering: 100%|██████████| 3500/3500 [00:02<00:00, 1270.90it/s]

   Filtering Results:
     - Total pairs: 3500
     - Method A mismatch: 271
     - Method B mismatch: 2449
     - Both match (Evaluated): 983 (28.1%)
   Evaluating accuracies using 21 workers...



Timeout during comparison
Timeout during comparison
Timeout during comparison
Timeout during comparison



📈 =================== COMPARISON SUMMARY ===================
   [EN] Acc A=7.14% (7/98) | Acc B=5.10% (5/98) | Delta=-2.04%
   [ES] Acc A=23.32% (104/446) | Acc B=24.89% (111/446) | Delta=+1.57%
   [ZH] Acc A=22.78% (100/439) | Acc B=20.50% (90/439) | Delta=-2.28%
   ----------------------------------------------------------
   [OVERALL] Acc A=21.46% (211/983) | Acc B=20.96% (206/983) | Delta=-0.51%



## 2. MATH-500 Comparison

In [4]:
evaluate_and_compare(
    benchmark_name="MATH-500-translated",
    prj_path=PRJ_PATH,
    model_path=MODEL_PATH,
    suffix_a=SUFFIX_A,
    suffix_b=SUFFIX_B
)


📊 Benchmark: MATH-500-translated
   Method A: Meta-Llama-3.1-8B-Instruct-cot.jsonl
   Method B: Meta-Llama-3.1-8B-Instruct-skeleton_multiturn.jsonl
   Loaded samples: A=3500, B=3000
🔍 Filtering language compliance pairs...


Filtering: 100%|██████████| 3000/3000 [00:01<00:00, 2093.19it/s]

   Filtering Results:
     - Total pairs: 3000
     - Method A mismatch: 90
     - Method B mismatch: 1935
     - Both match (Evaluated): 1034 (34.5%)
   Evaluating accuracies using 21 workers...



📈 =================== COMPARISON SUMMARY ===================
   [EN] Acc A=28.43% (29/102) | Acc B=22.55% (23/102) | Delta=-5.88%
   [ES] Acc A=37.85% (176/465) | Acc B=40.22% (187/465) | Delta=+2.37%
   [KO] Acc A=0.00% (0/1) | Acc B=0.00% (0/1) | Delta=+0.00%
   [ZH] Acc A=33.69% (157/466) | Acc B=37.12% (173/466) | Delta=+3.43%
   ----------------------------------------------------------
   [OVERALL] Acc A=35.01% (362/1034) | Acc B=37.04% (383/1034) | Delta=+2.03%



## 3. MGSM Comparison

In [5]:
evaluate_and_compare(
    benchmark_name="MGSM",
    prj_path=PRJ_PATH,
    model_path=MODEL_PATH,
    suffix_a=SUFFIX_A,
    suffix_b=SUFFIX_B
)


📊 Benchmark: MGSM
   Method A: Meta-Llama-3.1-8B-Instruct-cot.jsonl
   Method B: Meta-Llama-3.1-8B-Instruct-skeleton_multiturn.jsonl
   Loaded samples: A=2000, B=2000
🔍 Filtering language compliance pairs...


Filtering: 100%|██████████| 2000/2000 [00:00<00:00, 7690.37it/s]

   Filtering Results:
     - Total pairs: 2000
     - Method A mismatch: 6
     - Method B mismatch: 8
     - Both match (Evaluated): 1988 (99.4%)
   Evaluating accuracies using 21 workers...



📈 =================== COMPARISON SUMMARY ===================
   [BN] Acc A=60.64% (151/249) | Acc B=64.66% (161/249) | Delta=+4.02%
   [EN] Acc A=84.80% (212/250) | Acc B=84.80% (212/250) | Delta=+0.00%
   [ES] Acc A=76.00% (190/250) | Acc B=77.20% (193/250) | Delta=+1.20%
   [KO] Acc A=59.60% (149/250) | Acc B=64.00% (160/250) | Delta=+4.40%
   [SW] Acc A=62.08% (149/240) | Acc B=60.42% (145/240) | Delta=-1.67%
   [TE] Acc A=57.43% (143/249) | Acc B=63.86% (159/249) | Delta=+6.43%
   [TH] Acc A=66.40% (166/250) | Acc B=67.60% (169/250) | Delta=+1.20%
   [ZH] Acc A=72.40% (181/250) | Acc B=66.00% (165/250) | Delta=-6.40%
   ----------------------------------------------------------
   [OVERALL] Acc A=67.45% (1341/1988) | Acc B=68.61% (1364/1988) | Delta=+1.16%

